# PhyProbe — Physical Consistency Violation Scorer

Score any videos for physical consistency violations.  
**Score range:** `0.0` = no violation detected &rarr; `1.0` = strong violation.

> **Tip:** Use a GPU runtime for fast inference.  
> Runtime &rarr; Change runtime type &rarr; T4 GPU

In [ ]:
import subprocess

gpu_ok = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
if gpu_ok:
    print("GPU available ✅")
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
else:
    print("⚠️  No GPU detected — inference will run on CPU and may be slow.")
    print("   Go to: Runtime → Change runtime type → T4 GPU, then re-run.")

## 1. Install dependencies

`perception_models` (the PE vision backbone) is not on PyPI; it must be installed from GitHub.  
`decord` and `gdown` are the only other non-Colab deps.

In [ ]:
!pip install -q git+https://github.com/facebookresearch/perception_models.git
!pip install -q decord gdown
print("Installation complete.")

## 2. PhyProbe source (inlined)

The cell below defines the complete PhyProbe inference stack 

In [ ]:
# ============================================================
#  _preprocess.py — video loading
# ============================================================
from __future__ import annotations
from pathlib import Path
from typing import Optional, Union
import numpy as np
import torch

_DEFAULT_FPS = 16.0
# Gray fill for letterbox bars, as in training (pad_value=0.5).
_PAD_VALUE = 128


def _letterbox(frames, height, width):
    """Resize (T, H, W, 3) uint8 frames to fit (height, width), keeping aspect
    ratio and padding with gray.  Matches the training data loader exactly."""
    import torch.nn.functional as F
    t, h, w, c = frames.shape
    scale = min(width / max(w, 1), height / max(h, 1))
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    x = torch.from_numpy(frames).permute(0, 3, 1, 2).float()
    if (new_h, new_w) != (h, w):
        x = F.interpolate(x, size=(new_h, new_w), mode="bilinear", align_corners=False)
    x = x.clamp(0, 255).byte()
    canvas = torch.full((t, c, height, width), _PAD_VALUE, dtype=torch.uint8)
    top = (height - new_h) // 2
    left = (width - new_w) // 2
    canvas[:, :, top:top + new_h, left:left + new_w] = x
    return canvas  # (T, 3, H, W) uint8


def _load_video(
    path: Union[str, Path],
    num_frames: int = 48,
    height: int = 480,
    width: int = 480,
    fps: float = _DEFAULT_FPS,
    clip_sec: Optional[float] = None,
) -> torch.Tensor:
    """Load a video and return float32 [1, C, T, H, W] in [0, 1]."""
    from decord import VideoReader, cpu

    reader = VideoReader(str(path), ctx=cpu(0))
    total = len(reader)
    if total == 0:
        raise ValueError(f"No frames in video: {path}")

    native_fps = float(reader.get_avg_fps()) or 30.0
    duration_sec = total / native_fps

    n_resampled = max(1, int(round(duration_sec * fps)))
    resampled_indices = [
        min(total - 1, int(round((i / fps) * native_fps)))
        for i in range(n_resampled)
    ]

    if clip_sec is not None:
        max_frames = max(1, int(round(clip_sec * fps)))
        resampled_indices = resampled_indices[:max_frames]

    pool = resampled_indices
    n_pool = len(pool)

    if n_pool >= num_frames:
        positions = np.linspace(0, n_pool - 1, num_frames)
        sample_indices = [pool[int(round(p))] for p in positions]
    else:
        sample_indices = pool + [pool[-1]] * (num_frames - n_pool)

    frames = reader.get_batch(sample_indices).asnumpy()  # (T, H, W, 3) uint8
    del reader

    tensor = _letterbox(frames, height, width).permute(1, 0, 2, 3).float() / 255.0
    return tensor.unsqueeze(0)  # [1, 3, T, H, W]


# ============================================================
#  _backbone_pe.py — PE-based backbone + quality head
# ============================================================
import torch.nn as nn


def _import_pe():
    try:
        import core.vision_encoder.pe as pe
        from core.vision_encoder.config import PE_TEXT_CONFIG
        return pe, PE_TEXT_CONFIG
    except ImportError:
        raise ImportError(
            "perception_models is required.\n"
            "Install with: pip install git+https://github.com/facebookresearch/perception_models.git"
        )


def _rotate_half(x):
    d = x.shape[-1] // 2
    return torch.cat([-x[..., d:], x[..., :d]], dim=-1)


class _RoPE1D(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._cache_len = 0

    def _ensure_cache(self, length, device):
        if length <= self._cache_len:
            return
        pos = torch.arange(length * 2, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(pos, self.inv_freq.to(device))
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("_cos", emb.cos().unsqueeze(0).unsqueeze(0), persistent=False)
        self.register_buffer("_sin", emb.sin().unsqueeze(0).unsqueeze(0), persistent=False)
        self._cache_len = length * 2

    def forward(self, x, positions):
        self._ensure_cache(int(positions.max().item()) + 1, x.device)
        cos = self._cos[:, :, positions, :]
        sin = self._sin[:, :, positions, :]
        return x * cos + _rotate_half(x) * sin


class _RoPEAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model)
        self.rope = _RoPE1D(self.head_dim)

    def forward(self, x, positions):
        import torch.nn.functional as F
        B, T, D = x.shape
        def _h(t): return t.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.rope(_h(self.q_proj(x)), positions)
        K = self.rope(_h(self.k_proj(x)), positions)
        V = _h(self.v_proj(x))
        out = F.scaled_dot_product_attention(Q, K, V)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, D))


class _RoPETransformerLayer(nn.Module):
    def __init__(self, d_model, num_heads, dim_feedforward):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = _RoPEAttention(d_model, num_heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward), nn.GELU(),
            nn.Linear(dim_feedforward, d_model),
        )

    def forward(self, x, positions):
        x = x + self.attn(self.norm1(x), positions)
        x = x + self.ff(self.norm2(x))
        return x


class _RoPETransformerEncoder(nn.Module):
    def __init__(self, d_model, num_heads, num_layers, dim_feedforward):
        super().__init__()
        self.layers = nn.ModuleList([
            _RoPETransformerLayer(d_model, num_heads, dim_feedforward)
            for _ in range(num_layers)
        ])

    def forward(self, x, positions):
        for layer in self.layers:
            x = layer(x, positions)
        return x


def _make_sincos_embedding(max_len, dim):
    pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    div = torch.exp(
        torch.arange(0, dim, 2, dtype=torch.float32)
        * (-torch.log(torch.tensor(10000.0)) / dim)
    )
    pe = torch.zeros(1, max_len, dim)
    pe[0, :, 0::2] = torch.sin(pos * div)
    pe[0, :, 1::2] = torch.cos(pos * div[:dim // 2])
    return pe


class _PEModel(nn.Module):
    """
    PE backbone (PE-Core or PE-Spatial) + temporal-aggregation quality head.
    Input:  float32 [B, C, T, H, W] in [0, 1]
    Output: raw logit [B, 1]
    """
    def __init__(self, pe_config="PE-Core-G14-448", proj_head_mode="stats",
                 quality_head_dim=256, input_frames=None,
                 pos_embed_type="learned", device="cpu",
                 frame_chunk_size=4):
        super().__init__()
        pe, PE_TEXT_CONFIG = _import_pe()
        self.proj_head_mode = proj_head_mode
        self.pos_embed_type = pos_embed_type
        self._device = torch.device(device)
        self.frame_chunk_size = frame_chunk_size

        self._is_vision_only = pe_config not in PE_TEXT_CONFIG

        # Create the PE backbone directly in bfloat16 by temporarily overriding
        # the default dtype. Without this, from_config() allocates in float32
        # (~4.4 GB) and the .to(bfloat16) conversion peaks at ~6.6 GB.
        _prev_dtype = torch.get_default_dtype()
        torch.set_default_dtype(torch.bfloat16)
        try:
            if self._is_vision_only:
                self.pe_model = pe.VisionTransformer.from_config(pe_config, pretrained=False)
            else:
                self.pe_model = pe.CLIP.from_config(pe_config, pretrained=False)
        finally:
            torch.set_default_dtype(_prev_dtype)

        self.pe_model = self.pe_model.to(self._device).eval()
        for p in self.pe_model.parameters():
            p.requires_grad_(False)

        if hasattr(self.pe_model, "visual") and hasattr(self.pe_model.visual, "embed_dim"):
            self.feature_dim = self.pe_model.visual.embed_dim
        elif hasattr(self.pe_model, "embed_dim"):
            self.feature_dim = self.pe_model.embed_dim
        else:
            with torch.no_grad():
                dummy = torch.randn(1, 3, 224, 224, device=self._device, dtype=torch.bfloat16)
                out = self.pe_model(dummy) if self._is_vision_only else self.pe_model.encode_image(dummy)
                if out.dim() == 3:
                    out = out.mean(1)
                self.feature_dim = out.shape[-1]

        if proj_head_mode == "stats":
            head_in_dim = self.feature_dim * 3
        elif proj_head_mode == "attn_pool":
            head_in_dim = self.feature_dim
            self.attn_query = nn.Parameter(torch.zeros(1, 1, self.feature_dim))
            nn.init.trunc_normal_(self.attn_query, std=0.02)
            self.attn_pool = nn.MultiheadAttention(self.feature_dim, num_heads=16, batch_first=True)
        elif proj_head_mode == "transformer":
            head_in_dim = self.feature_dim
            enc_layer = nn.TransformerEncoderLayer(
                d_model=self.feature_dim, nhead=16, dim_feedforward=self.feature_dim * 4,
                dropout=0.0, activation="gelu", batch_first=True, norm_first=True,
            )
            self.transformer_head = nn.TransformerEncoder(enc_layer, num_layers=2)
        elif proj_head_mode == "temporal_transformer":
            head_in_dim = self.feature_dim
            self.temporal_cls = nn.Parameter(torch.zeros(1, 1, self.feature_dim))
            nn.init.trunc_normal_(self.temporal_cls, std=0.02)
            if pos_embed_type == "rope":
                self.temporal_transformer = _RoPETransformerEncoder(
                    d_model=self.feature_dim, num_heads=16,
                    num_layers=2, dim_feedforward=self.feature_dim * 4,
                )
            else:
                max_t = (input_frames or 64) + 1
                if pos_embed_type == "sincos":
                    self.register_buffer(
                        "temporal_pos_embed", _make_sincos_embedding(max_t, self.feature_dim)
                    )
                else:
                    self.temporal_pos_embed = nn.Parameter(torch.zeros(1, max_t, self.feature_dim))
                    nn.init.trunc_normal_(self.temporal_pos_embed, std=0.02)
                enc_layer = nn.TransformerEncoderLayer(
                    d_model=self.feature_dim, nhead=16, dim_feedforward=self.feature_dim * 4,
                    dropout=0.0, activation="gelu", batch_first=True, norm_first=True,
                )
                self.temporal_transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
        else:  # mlp
            head_in_dim = self.feature_dim

        self.quality_head = nn.Sequential(
            nn.Linear(head_in_dim, quality_head_dim), nn.GELU(),
            nn.Linear(quality_head_dim, quality_head_dim // 2), nn.GELU(),
            nn.Linear(quality_head_dim // 2, 1),
        )

    def _encode_frame(self, frames):
        if self._is_vision_only:
            out = self.pe_model(frames)
            if out.dim() == 3:
                out = out.mean(dim=1)
        else:
            out = self.pe_model.encode_image(frames)
        return out.float()

    def _pool_features(self, features):
        if self.proj_head_mode == "mlp":
            return features.mean(dim=1)
        if self.proj_head_mode == "stats":
            return torch.cat([
                features.mean(dim=1),
                features.std(dim=1, unbiased=False),
                features.max(dim=1).values,
            ], dim=-1)
        if self.proj_head_mode == "attn_pool":
            q = self.attn_query.expand(features.size(0), -1, -1)
            out, _ = self.attn_pool(q, features, features)
            return out.squeeze(1)
        if self.proj_head_mode == "transformer":
            return self.transformer_head(features).mean(dim=1)
        if self.proj_head_mode == "temporal_transformer":
            B, T, _ = features.shape
            cls = self.temporal_cls.expand(B, -1, -1)
            x = torch.cat([cls, features], dim=1)
            if self.pos_embed_type == "rope":
                positions = torch.arange(T + 1, device=x.device)
                x = self.temporal_transformer(x, positions)
            else:
                x = x + self.temporal_pos_embed[:, :T + 1]
                x = self.temporal_transformer(x)
            return x[:, 0]
        raise ValueError(f"Unknown proj_head_mode: {self.proj_head_mode!r}")

    def forward(self, video):
        B, C, T, H, W = video.shape
        frames = video.transpose(1, 2).contiguous().view(B * T, C, H, W).to(torch.bfloat16)

        # Encode in chunks: pushing all 48 frames through PE-Core at once fills
        # ~9 GB of VRAM with activations. frame_chunk_size=4 keeps the peak
        # to ~0.75 GB of activations without affecting scores (frames are independent).
        chunk_feats = []
        for i in range(0, frames.shape[0], self.frame_chunk_size):
            chunk_feats.append(self._encode_frame(frames[i : i + self.frame_chunk_size]))
        feats = torch.cat(chunk_feats, dim=0).view(B, T, -1)

        return self.quality_head(self._pool_features(feats))


# ============================================================
#  _hub.py / __init__.py — checkpoint download + PhyProbe class
# ============================================================
import gc
import os
import torch

# Reduce CUDA allocator fragmentation — helps when VRAM is tight.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

_CHECKPOINT_URL_GDRIVE = "1SdGpDY3TDeML5KRUviEgJ2z7yEw-5Vw2"
_CHECKPOINT_CONFIG = {
    "model_backbone": "pecore_proj",
    "pecore_variant": "PE-Core-G14-448",
    "pecore_proj_head_mode": "stats",
    "quality_head_dim": 256,
    "cur_frames": 48,
    "cur_height": 480,
    "cur_width": 480,
    "cur_target_fps": 16.0,
}


def _download_checkpoint(dest_path):
    import gdown
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if dest_path.exists():
        print(f"Checkpoint already cached at {dest_path}")
        return dest_path
    print(f"Downloading checkpoint from Google Drive → {dest_path} ...")
    gdown.download(id=_CHECKPOINT_URL_GDRIVE, output=str(dest_path), quiet=False)
    return dest_path


def _load_checkpoint(ckpt_path, device="cpu"):
    # mmap=True (PyTorch >= 2.1): memory-map the file so tensors are paged in
    # during load_state_dict rather than all read into RAM at once.
    try:
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=True, mmap=True)
    except TypeError:
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    if isinstance(ckpt, dict) and "config" in ckpt:
        config = ckpt["config"]
        key = "ema_model_state_dict" if "ema_model_state_dict" in ckpt else "model_state_dict"
        state = ckpt[key]
    else:
        config = dict(_CHECKPOINT_CONFIG)
        state = ckpt
    state = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in state.items()}
    return config, state


class PhyProbe:
    """
    Physical consistency violation scorer for generated videos.
    score: 0 = no violation, 1 = strong violation.
    """

    def __init__(self, checkpoint_path="/content/phyprobe_ckpt/pecore_paper.pt",
                 device=None, frame_chunk_size=4, compile_model=False):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = torch.device(device)

        ckpt_path = _download_checkpoint(checkpoint_path)
        config, state = _load_checkpoint(ckpt_path, "cpu")

        self._frames = config.get("cur_frames", 48)
        self._height = config.get("cur_height", 480)
        self._width  = config.get("cur_width",  480)
        self._fps    = float(config.get("cur_target_fps", config.get("cur_fps", 16.0)))

        self._model = _PEModel(
            pe_config=config.get("pecore_variant", "PE-Core-G14-448"),
            proj_head_mode=config.get("pecore_proj_head_mode", "stats"),
            quality_head_dim=config.get("quality_head_dim", 256),
            input_frames=self._frames,
            pos_embed_type=config.get("pecore_pos_embed_type", "learned"),
            device="cpu",
            frame_chunk_size=frame_chunk_size,
        )
        self._model.load_state_dict(state, strict=True)

        del state
        gc.collect()

        self._model.eval().to(self.device)

        if self.device.type == "cuda":
            torch.cuda.empty_cache()

        # torch.compile with "default" mode avoids CUDA Graphs (used by
        # "reduce-overhead"), which conflict with the RoPE buffer updates in
        # perception_models and cause errors on re-inference.
        # "default" still fuses ops via Triton and gives a meaningful speedup
        # after the first-call compilation (~1-2 min warmup).
        if compile_model:
            print("Compiling model with torch.compile (first inference will be slow — this is normal)...")
            self._model = torch.compile(self._model, mode="default")

    @torch.inference_mode()
    def score(self, video_path, clip_sec=None):
        """Score a single video. Returns raw score clamped to [0, 1]."""
        tensor = _load_video(
            video_path, num_frames=self._frames, height=self._height,
            width=self._width, fps=self._fps, clip_sec=clip_sec,
        ).to(self.device)
        use_cuda = self.device.type == "cuda"
        ctx = torch.amp.autocast("cuda", dtype=torch.bfloat16) if use_cuda else torch.amp.autocast("cpu")
        with ctx:
            raw = self._model(tensor).squeeze()
        return torch.clamp(raw.float(), 0.0, 1.0).item()

    @torch.inference_mode()
    def score_batch(self, video_paths, clip_sec=None):
        """Score a list of video paths. Returns list of raw scores clamped to [0, 1]."""
        use_cuda = self.device.type == "cuda"
        ctx = torch.amp.autocast("cuda", dtype=torch.bfloat16) if use_cuda else torch.amp.autocast("cpu")
        scores = []
        for path in video_paths:
            tensor = _load_video(
                path, num_frames=self._frames, height=self._height,
                width=self._width, fps=self._fps, clip_sec=clip_sec,
            ).to(self.device)
            with ctx:
                raw = self._model(tensor).squeeze()
            scores.append(torch.clamp(raw.float(), 0.0, 1.0).item())
        return scores


print("PhyProbe source loaded ✅")

## 3. Load the model

On the first run the pretrained checkpoint (~5 GB) is downloaded automatically from Google Drive  
and cached at `/content/phyprobe_ckpt/pecore_paper.pt`.

In [ ]:
model = PhyProbe()
print(f"Model loaded on device: {model.device}")
print(f"Config: {model._frames} frames @ {model._fps:g} fps, {model._height}×{model._width}px")

## 4. Upload and score videos

Upload one or two video files.  
If two videos are provided, the notebook also reports which one has a higher violation score.

In [ ]:
import os, tempfile

def _verdict(s):
    if s < 0.2:
        return "🟢 no to low violation"
    elif s < 0.6:
        return "🟡 moderate violation"
    else:
        return "🔴 high violation"

try:
    from google.colab import files as colab_files
    print("Select one or two video files:")
    uploaded = colab_files.upload()  # {filename: bytes}
except ImportError:
    uploaded = {}
    print("Not in Colab — set video_paths manually below.")

video_paths, video_names = [], []
for name, data in uploaded.items():
    ext = os.path.splitext(name)[1] or ".mp4"
    tmp = tempfile.NamedTemporaryFile(suffix=ext, delete=False)
    tmp.write(data); tmp.flush()
    video_paths.append(tmp.name)
    video_names.append(name)

# ── Or set paths manually when not in Colab ───────────────────────────
# video_paths = ["/path/to/video_a.mp4", "/path/to/video_b.mp4"]
# video_names = ["video_a.mp4", "video_b.mp4"]

if not video_paths:
    print("No videos provided.")
else:
    scores = model.score_batch(video_paths)
    print("\n" + "─" * 56)
    for name, s in zip(video_names, scores):
        print(f"{name}:  {s:.4f}  —  {_verdict(s)}")
    if len(scores) == 2:
        delta = scores[0] - scores[1]
        more = video_names[0] if delta > 0 else video_names[1]
        print(f"\nΔ (A − B) = {delta:+.4f}  →  more violated: {more}")
    print("─" * 56)
    print("Score: 0.0–0.2 = no/low  |  0.2–0.6 = moderate  |  0.6–1.0 = high violation")

## 5. (Optional) Score only the first N seconds

The model was trained on the first 3 seconds of each video (48 frames @ 16 fps); use `CLIP_SEC = 3.0` to reproduce the paper's numbers.

In [ ]:
CLIP_SEC = 3.0   # adjust as needed

if video_paths:
    scores_clipped = model.score_batch(video_paths, clip_sec=CLIP_SEC)
    print(f"Scores (first {CLIP_SEC:g} s only):")
    for name, s in zip(video_names, scores_clipped):
        print(f"  {_verdict(s)}  {name}: {s:.4f}")

## 6. (Optional) Interactive Gradio UI

Launches a web interface with `share=True` — you get a public link valid for 72 hours.

In [ ]:
!pip install -q "gradio<5.49"

import gradio as gr

_UNIFORM = "Uniform sampling (full video)"
_FIRST_N = "First N seconds only"


def _verdict(s):
    if s < 0.2:
        return "🟢 no to low violation"
    elif s < 0.6:
        return "🟡 moderate violation"
    else:
        return "🔴 high violation"


def _run_ui(video_a, video_b, sampling_mode, clip_sec_val):
    if video_a is None:
        raise gr.Error("Upload at least one video (Video A).")
    clip = float(clip_sec_val) if sampling_mode == _FIRST_N else None
    videos = [video_a] + ([video_b] if video_b else [])
    scores = model.score_batch(videos, clip_sec=clip)
    desc = f"first {clip:g} s" if clip else "full video"
    lines = [f"_Sampling: {desc}, uniformly sampled → {model._frames} frames @ {model._fps:g} fps_", ""]
    lines.append(f"**Video A**: `{scores[0]:.4f}`  —  {_verdict(scores[0])}")
    if len(scores) == 2:
        lines.append(f"**Video B**: `{scores[1]:.4f}`  —  {_verdict(scores[1])}")
        delta = scores[0] - scores[1]
        more = "Video A" if delta > 0 else "Video B"
        lines.append(f"\n**Δ (A − B)** = `{delta:+.4f}`  →  more violated: **{more}**")
    lines.append("\n_Score: 0.0–0.2 = no/low  |  0.2–0.6 = moderate  |  0.6–1.0 = high violation_")
    return "\n\n".join(lines)


with gr.Blocks(title="PhyProbe Demo") as demo:
    gr.Markdown(
        "# 🧪 PhyProbe — Physical Consistency Violation Scorer\n"
        "Upload one video to score it, or two videos to compare."
    )
    with gr.Row():
        vid_a = gr.Video(label="Video A (required)")
        vid_b = gr.Video(label="Video B (optional — for comparison)")
    with gr.Row():
        mode = gr.Radio(choices=[_UNIFORM, _FIRST_N], value=_UNIFORM, label="Frame sampling")
        clip_box = gr.Number(value=3.0, precision=1, minimum=0.5, maximum=10.0, step=0.5,
                             label="Clip length (seconds)", visible=False)
    mode.change(lambda m: gr.update(visible=(m == _FIRST_N)), inputs=mode, outputs=clip_box)
    btn = gr.Button("Score", variant="primary")
    out = gr.Markdown()
    btn.click(_run_ui, inputs=[vid_a, vid_b, mode, clip_box], outputs=out)

demo.queue().launch(share=True)